### Initialize the Environment:

##### Virtual Environment Commands

| Command | Linux/Mac | GitBash |
| ------- | --------- | ------- |
| Create | `python3 -m venv venv` | `python -m venv venv` |
| Activate | `source venv/bin/activate` | `source venv/Scripts/activate` |
| Install | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| Deactivate | `deactivate` | `deactivate` |

##### Select the Kernel (This will be in the Requirements.txt eventually)

Using the venv (Python 3.13.2) located  in venv/bin/python)


### **Project Overview & Plan**

Capstone Project for Code:You Data Analysis track. This project analyzes Beer Recipes for frequency of uploads for various beer styles, while capturing preferences of strength, hopiness and batch size.    The goal of the project is to demonstrate a general knowledge of Python (Pandas, Numpy, MatLibPlot, Plotly), SQL(MySQL), Tableu, Cursor and ChatGPT.

**Data Sources:**

The datasets used in this project are all related to online beer recipes. One dataset contains the different styles of beer as recognized by the Beer Judge Certification Program (BJCP.org).
- [Beersmith Recipes](https://beersmithrecipes.com/recent/) - scraped data that contains certain fields of 100% of the all grain beer recipes that have been uploaded by users.

- [Brewers Friend All-Grain Recipes](https://www.brewersfriend.com/homebrew-recipes/all-grain/) - scraped data that contains select fields of 100% of the all-grain beer recipes that have been uploaded by users.
- [Kaggle - Brewers Friend Recipes](https://www.kaggle.com/datasets/jtrofe/beer-recipes) - data from Kaggle that contains a subset of beer recipes.
- [BJCP - Judging Styles of Beer](https://github.com/ascholer/bjcp-styleview/blob/main/styles.json) - dataset that contains the criterea used to judge beer. Will help determine if recipes meet the criterea to be considered a specific style of beer.

**Cleaning Methodology:**
I will be doing some preliminary cleaning of each dataset to create commonality between the datasets as far as column names and column content is concerned. Once the individual data sets are common, then I will use common functions to clean the datasets.

In [ ]:
import pandas as pd
# import matplotlib
from pandas import DataFrame
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
# from rich.console import Console
# from rich.table import Table

Note: styles.json from: https://github.com/ascholer/bjcp-styleview

### **Import Data** and preview Data shapes

In [31]:
# bs_df = pd.read_csv('beersmith_recipes.csv')
bf_df = pd.read_csv('bf_recipes.csv')
# kag_df = pd.read_csv('recipeData.csv', encoding='ISO-8859-1')
styles_df = pd.read_json('styles.json')
print(bf_df.shape)
# print(bs_df.shape)
# print(kag_df.shape)
print(styles_df.shape)

(215580, 8)
(116, 28)


### Data Cleanup

#### **Brewers Friend Recipes -** bf_recipes.csv = bf_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data and rename columns to match up with the BeerSmith column names. In the Brewers friend data there are 14 null values in the title column. Since the name isn't crucial to the analysis of the recipes, but the recipe information is still of value, I  assigned a generic name to each of the rows missing the Title.

In [32]:
print(bf_df.info())
print(bf_df.columns)
# print(kag_df.info())
# print(styles_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215580 entries, 0 to 215579
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Title   215566 non-null  object 
 1   Style   215580 non-null  object 
 2   Size    215580 non-null  object 
 3   OG      215580 non-null  float64
 4   FG      215580 non-null  float64
 5   ABV     215580 non-null  float64
 6   IBU     215580 non-null  float64
 7   Color   215580 non-null  object 
dtypes: float64(4), object(4)
memory usage: 13.2+ MB
None
Index(['Title', 'Style', 'Size', 'OG', 'FG', 'ABV', 'IBU', 'Color'], dtype='object')


In [33]:
def clean_recipe_data(df, columns_to_drop, column_mapping, location, column_name, formula, location_2, column_name_2, new_column_2 ) -> pd.DataFrame:
    """
    Part of the cleanup:
        Creating a copy of dataframe with cleaned up data
        Removing unneeded columns
        Renaming columns to match with other dataframes
        Adding column with complex calculation
        

    Parameters:
    df: The Dataframes we are dropping columns
    columns_to_drop: the columns we don't need
    columns_to_rename: the columns to rename so they are common between the 2 data sets


    """
    # List of columns to drop by index
    # cols_to_drop = [2, 7, ]

    # Drop the specified columns
    df.drop(df.columns[columns_to_drop], axis=1, inplace=True)
    
    # Adding a new column named with a formula to create calculated value
    # match up with the other dataframes
    df.rename(columns=column_mapping, inplace=True)
    
    # Calculate the new column
    new_column = df.eval(formula)
    
    # Round up to 1 decimal place
    new_column = new_column.apply(lambda x: round(x, 1))
    
    # Insert the new column at the specified location with calculated value
    df.insert(location, column_name, new_column)
    
    # Insert another new column with no data for future calculations
    df.insert(location_2, column_name_2, new_column_2)
    # print(df.info)

    return df

In [34]:
# Columns to drop in bf_df_cleaned
cols_to_drop = [2, 7 ]

# Columns to rename and the new name
column_mapping = {
    'Title': 'Recipe Name'
}

# Add column for plato and calculate values to populate
location = 3
column_name = 'Plato'
formula = '-463.37 + (668.72 * OG) - (205.35 * OG**2)'

# Add column to match other recipe dataset
location_2 = 2
column_name_2 = 'Style Number'
new_column_2 = None


bf_df_cleaned = clean_recipe_data(
    bf_df, cols_to_drop, column_mapping, location, column_name,
    formula, location_2, column_name_2, new_column_2
)
bf_df_cleaned


,Recipe Name,Style,Style Number,OG,Plato,FG,ABV,IBU
0,Avg. Perfect Northeast IPA (NEIPA),Specialty IPA: New England IPA,None,1.062,15.2,1.013,6.50,59.26
1,Sierra Nevada Pale Ale Clone,American Pale Ale,None,1.055,13.6,1.013,5.58,39.79
2,Vanilla Cream Ale,Cream Ale,None,1.055,13.6,1.013,5.48,19.44
3,Zombie Dust Clone - ALL GRAIN,American IPA,None,1.061,15.0,1.016,5.94,62.42
4,Russian River Pliny The Elder (original),Imperial IPA,None,1.072,17.5,1.018,7.09,232.89
...,...,...,...,...,...,...,...,...
215575,Awesome Recipe,American IPA,None,1.054,13.3,1.010,5.82,19.23
215576,Dildo NEIPA,Specialty IPA: New England IPA,None,1.061,15.0,1.012,6.43,18.70
215577,Double Down,Specialty IPA: New England IPA,None,1.082,19.8,1.015,8.83,28.76
215578,AGAIN,American IPA,None,1.029,7.3,1.010,2.51,60.87


In [35]:
# Function to delete rows that don't contain information or have incorrect values
# Incorrect values are values that aren't possible. For example can't have an OG > 1.250
def delete_useless_rows(df, column_name, condition):
    # This function will delete rows that contain unusable data
    if isinstance(condition, str):
        # For string conditions, use str.contains
        df = df[~df[column_name].str.contains(condition, case=False, na=False)]
    else :
        df = df[~df[column_name].apply(condition)]
    return df

   
    

In [ ]:
# Parameters for removing useless rows

# Removing when there is no style to work with
column_name = 'Style'
condition = 'No Profile Selected' # type: ignore
bf_df_cleaned = delete_useless_rows(bf_df_cleaned, column_name, condition)

# Removing rows when the data is out of range
column_name = 'OG'
def condition(x):
    return x > 1.25
bf_df_cleaned = delete_useless_rows(bf_df_cleaned, column_name, condition)

bf_df_cleaned

,Recipe Name,Style,Style Number,OG,Plato,FG,ABV,IBU
0,Avg. Perfect Northeast IPA (NEIPA),Specialty IPA: New England IPA,None,1.062,15.2,1.013,6.50,59.26
1,Sierra Nevada Pale Ale Clone,American Pale Ale,None,1.055,13.6,1.013,5.58,39.79
2,Vanilla Cream Ale,Cream Ale,None,1.055,13.6,1.013,5.48,19.44
3,Zombie Dust Clone - ALL GRAIN,American IPA,None,1.061,15.0,1.016,5.94,62.42
4,Russian River Pliny The Elder (original),Imperial IPA,None,1.072,17.5,1.018,7.09,232.89
...,...,...,...,...,...,...,...,...
215575,Awesome Recipe,American IPA,None,1.054,13.3,1.010,5.82,19.23
215576,Dildo NEIPA,Specialty IPA: New England IPA,None,1.061,15.0,1.012,6.43,18.70
215577,Double Down,Specialty IPA: New England IPA,None,1.082,19.8,1.015,8.83,28.76
215578,AGAIN,American IPA,None,1.029,7.3,1.010,2.51,60.87


In [6]:
null_title_rows = bf_df_cleaned[bf_df_cleaned['Recipe Name'].isnull()]

# Display the 14 null rows in the Title column
print(null_title_rows.head(14))

       Recipe Name                   Style Style Number     OG  Plato     FG  \
90398          NaN       American Pale Ale         None  1.051   12.6  1.010   
104343         NaN     American Barleywine         None  1.157   35.4  1.039   
134732         NaN            American IPA         None  1.001    0.3  1.000   
141240         NaN     No Profile Selected         None  1.018    4.6  1.015   
141740         NaN             Sweet Stout         None  1.059   14.5  1.017   
143956         NaN  Russian Imperial Stout         None  1.117   27.4  1.031   
145756         NaN     No Profile Selected         None  1.063   15.4  1.016   
149102         NaN     No Profile Selected         None  1.058   14.3  1.015   
149370         NaN       American Pale Ale         None  1.041   10.2  1.010   
154188         NaN            American IPA         None  1.062   15.2  1.014   
170141         NaN       American Pale Ale         None  1.056   13.8  1.011   
195556         NaN              Blonde A

In [ ]:
# This function is to replace null values in the Recipe Name Column with a Generic
# unique name that begins with the first 2 characters of the dataframe name and is 
# incremented by 1 to keep the name unique. This was done because the data in the 
# rest of the columns contributed to the data and analysis.
# This code was written with a lot of back and forth with Perplexity.ai.

import inspect

def replace_column_nulls(df, target_column):
    # Replace nulls in one specific column with unique numbered values
    # Get dataframe variable name safely
    try:
        caller_frame = inspect.currentframe().f_back  # type: ignore
        df_name = [k for k, v in caller_frame.f_locals.items() if v is df][0]  # type: ignore
        prefix = df_name[:2].upper()
    except (AttributeError, IndexError):  # Catch only relevant errors
        prefix = "DF"

    # Create column-specific generator
    def col_generator():
        counter = 1
        while True:
            yield f"{prefix} No Name {counter}"
            counter += 1

    # Only process specified column
    mask = df[target_column].isnull()
    num_nulls = mask.sum()
    
    if num_nulls > 0:
        gen = col_generator()
        replacements = [next(gen) for _ in range(num_nulls)]
        df.loc[mask, target_column] = replacements
        
        # Show changes
        print(f"Replaced {num_nulls} nulls in {target_column}")
        print("Example replacement:", replacements[0])
    
    return df



In [8]:
bf_df_cleaned = replace_column_nulls(bf_df_cleaned, 'Recipe Name')


Replaced 14 nulls in Recipe Name
Example replacement: BF No Name 1


#### **Brewers Freind Recipes -** bf_recipes.csv = bf_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data, split the stats column into the individual components to match up with the Beersmith data. While cleaning up the data, will create a function to clean up the data in the other datasets. In the Beer Smith data there were 31 rows missing the Recipe Name. Since the name wasn't crucial to the analysis of the recipes, but the recipe information was still of value, we assigned a generic name to each of the rows missing the Recipe Name.

In [10]:
# Brewers Friend dataframe info
print(bf_df.info())
print(bf_df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215580 entries, 0 to 215579
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Recipe Name   215580 non-null  object 
 1   Style         215580 non-null  object 
 2   Style Number  0 non-null       object 
 3   OG            215580 non-null  float64
 4   Plato         215580 non-null  float64
 5   FG            215580 non-null  float64
 6   ABV           215580 non-null  float64
 7   IBU           215580 non-null  float64
dtypes: float64(5), object(3)
memory usage: 13.2+ MB
None
Index(['Recipe Name', 'Style', 'Style Number', 'OG', 'Plato', 'FG', 'ABV',
       'IBU'],
      dtype='object')


In [ ]:
# Additional specific cleanup - didn't realize there were items that weren't null
# But were still missing information when the Style column had 'No Profile Specified"